In [2]:
import numpy as np
from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 1 - WEEK 9 BAYESIAN OPTIMISATION
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function1/initial_inputs.npy")
Y = np.load("function1/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nObserved Y range:")
print("min =", Y.min())
print("max =", Y.max())


# ------------------------------------------------------------
# 2. Review Week 8 surrogate behaviour
# ------------------------------------------------------------
#
# Week 8 GP prediction at submitted candidate:
# mean ≈ 8.90e-05
# std  ≈ 1.46e-04
#
# Actual Week 8 result:
# 2.07e-54
#
# The GP predictive scale was many orders of magnitude larger
# than the entire useful observed objective range.
#
# For Week 9 we therefore use a POSITIVE LINEAR scaling of Y.
# This changes numerical scale only and preserves argmax exactly.
# ------------------------------------------------------------

week8_pred_mean = 8.902903800467573e-05
week8_pred_std = 1.4631037595458956e-04
week8_actual = 2.0668348995908508e-54

print("\nWeek 8 prediction check:")
print("Predicted mean =", week8_pred_mean)
print("Predicted std  =", week8_pred_std)
print("Actual         =", week8_actual)

print(
    "Predicted mean / historical best magnitude =",
    abs(week8_pred_mean) / max(abs(best_y), 1e-300)
)


# ------------------------------------------------------------
# 3. Linear objective scaling
# ------------------------------------------------------------

y_scale = np.max(np.abs(Y))

if y_scale == 0:
    y_scale = 1.0

Y_scaled = Y / y_scale
best_y_scaled = best_y / y_scale

print("\nLinear Y scale:", y_scale)
print("Scaled Y range:", Y_scaled.min(), "to", Y_scaled.max())
print("Scaled best:", best_y_scaled)


# ------------------------------------------------------------
# 4. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(2) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=False,          # already scaled manually
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y_scaled)

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 5. Expected Improvement function
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 6. GLOBAL diagnostic search
# ------------------------------------------------------------
#
# Function 1 is only 2D, so we can cheaply examine the
# whole domain. This is diagnostic rather than automatically
# accepting the global recommendation.
# ------------------------------------------------------------

axis = np.linspace(0, 1, 401)

g1, g2 = np.meshgrid(axis, axis)

global_candidates = np.column_stack([
    g1.ravel(),
    g2.ravel()
])

tree = cKDTree(X)

distance, _ = tree.query(global_candidates, k=1)

global_candidates = global_candidates[
    distance > 0.01
]

g_mu, g_sigma = gp.predict(
    global_candidates,
    return_std=True
)

g_EI = expected_improvement(
    g_mu,
    g_sigma,
    best_y_scaled,
    xi=0.0
)

g_ei_idx = np.argmax(g_EI)
g_mean_idx = np.argmax(g_mu)

print("\n================================")
print("GLOBAL DIAGNOSTICS")
print("================================")

print("\nGlobal EI:")
print("candidate =", global_candidates[g_ei_idx])
print("scaled mean =", g_mu[g_ei_idx])
print("scaled std =", g_sigma[g_ei_idx])
print("EI =", g_EI[g_ei_idx])

print("\nGlobal highest mean:")
print("candidate =", global_candidates[g_mean_idx])
print("scaled mean =", g_mu[g_mean_idx])
print("scaled std =", g_sigma[g_mean_idx])


# ------------------------------------------------------------
# 7. Data-driven trust region
# ------------------------------------------------------------
#
# Instead of manually choosing the radius:
#
# 1. centre automatically on current best
# 2. find distance to its nearest observed neighbour
# 3. allow up to 2x that empirical distance
# 4. combine that cap with fitted ARD lengthscales
#
# This contracts the search automatically because F1 has shown
# poor global extrapolation.
# ------------------------------------------------------------

best_distances = np.linalg.norm(
    X - best_x,
    axis=1
)

nonzero_distances = best_distances[
    best_distances > 1e-12
]

nearest_distance = np.min(nonzero_distances)

empirical_cap = min(
    2.0 * nearest_distance,
    0.15
)

trust_half_width = np.clip(
    0.5 * lengthscales,
    0.01,
    empirical_cap
)

lower = np.maximum(
    0.0,
    best_x - trust_half_width
)

upper = np.minimum(
    1.0,
    best_x + trust_half_width
)

print("\n================================")
print("TRUST REGION")
print("================================")

print("Nearest-neighbour distance:", nearest_distance)
print("Empirical width cap:", empirical_cap)
print("Trust half-widths:", trust_half_width)

print("Lower bounds:", lower)
print("Upper bounds:", upper)


# ------------------------------------------------------------
# 8. Dense trust-region search
# ------------------------------------------------------------

x1_axis = np.linspace(
    lower[0],
    upper[0],
    601
)

x2_axis = np.linspace(
    lower[1],
    upper[1],
    601
)

t1, t2 = np.meshgrid(
    x1_axis,
    x2_axis
)

tr_candidates = np.column_stack([
    t1.ravel(),
    t2.ravel()
])

distance, _ = tree.query(
    tr_candidates,
    k=1
)

tr_candidates = tr_candidates[
    distance > 0.01
]

tr_mu, tr_sigma = gp.predict(
    tr_candidates,
    return_std=True
)

tr_EI = expected_improvement(
    tr_mu,
    tr_sigma,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(tr_EI)
mean_idx = np.argmax(tr_mu)


# ------------------------------------------------------------
# 9. Trust-region diagnostics
# ------------------------------------------------------------

print("\n================================")
print("TRUST-REGION RESULTS")
print("================================")

print("\nPRIMARY EI:")
print("candidate =", tr_candidates[ei_idx])
print("scaled mean =", tr_mu[ei_idx])
print("scaled std =", tr_sigma[ei_idx])
print("EI =", tr_EI[ei_idx])

print("\nHighest predicted mean:")
print("candidate =", tr_candidates[mean_idx])
print("scaled mean =", tr_mu[mean_idx])
print("scaled std =", tr_sigma[mean_idx])

print("\nUCB diagnostics:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = tr_mu + beta * tr_sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", tr_candidates[idx],
        "\n scaled mean =", tr_mu[idx],
        "\n scaled std =", tr_sigma[idx],
        "\n UCB =", UCB[idx],
        "\n"
    )

X shape: (18, 2)
Y shape: (18,)

Current best:
[0.73102363 0.73299988] -> 7.710875114502849e-16

Observed Y range:
min = -0.0036060626443634764
max = 7.710875114502849e-16

Week 8 prediction check:
Predicted mean = 8.902903800467573e-05
Predicted std  = 0.00014631037595458956
Actual         = 2.0668348995908508e-54
Predicted mean / historical best magnitude = 115459058385.2502

Linear Y scale: 0.0036060626443634764
Scaled Y range: -1.0 to 2.1383086970370515e-13
Scaled best: 2.1383086970370515e-13


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 2.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



Fitted kernel:
0.254**2 * Matern(length_scale=[2, 0.0346], nu=2.5) + WhiteKernel(noise_level=1e-08)

ARD lengthscales:
[2.         0.03458915]

Normalised inverse-lengthscale sensitivity:
[0.01700056 0.98299944]

GLOBAL DIAGNOSTICS

Global EI:
candidate = [0.   0.52]
scaled mean = 0.005079602215344044
scaled std = 0.24976322860823225
EI = 0.10220151919873495

Global highest mean:
candidate = [0.9325 0.56  ]
scaled mean = 0.015804672610714086
scaled std = 0.1705141843448853

TRUST REGION
Nearest-neighbour distance: 0.026278458561148094
Empirical width cap: 0.05255691712229619
Trust half-widths: [0.05255692 0.01729457]
Lower bounds: [0.67846671 0.7157053 ]
Upper bounds: [0.78358055 0.75029445]

TRUST-REGION RESULTS

PRIMARY EI:
candidate = [0.67846671 0.73732352]
scaled mean = 0.006480539558408549
scaled std = 0.009478077682684926
EI = 0.007872444298328715

Highest predicted mean:
candidate = [0.67846671 0.73697763]
scaled mean = 0.006510847743953774
scaled std = 0.00929783193960698

UC

In [3]:
# ============================================================
# FUNCTION 1 WEEK 9 - ONE-DIMENSION TRUST-REGION EXPANSION
# ============================================================
#
# All trust-region acquisition criteria hit the LOWER x1 boundary.
# Expand x1 once, while leaving x2 unchanged.
#
# This is justified because:
# - x1 lengthscale is at the upper bound (very smooth in GP fit)
# - x2 lengthscale is short and should remain tightly constrained
# - the expansion is triggered automatically by a boundary hit

expanded_half_width = trust_half_width.copy()

# Expand only x1 by a factor of 2
expanded_half_width[0] = min(
    2.0 * trust_half_width[0],
    0.15
)

expanded_lower = np.maximum(
    0.0,
    best_x - expanded_half_width
)

expanded_upper = np.minimum(
    1.0,
    best_x + expanded_half_width
)

print("Original half-widths:", trust_half_width)
print("Expanded half-widths:", expanded_half_width)

print("\nExpanded lower bounds:", expanded_lower)
print("Expanded upper bounds:", expanded_upper)


# ------------------------------------------------------------
# Dense search in expanded region
# ------------------------------------------------------------

x1_axis = np.linspace(
    expanded_lower[0],
    expanded_upper[0],
    801
)

x2_axis = np.linspace(
    expanded_lower[1],
    expanded_upper[1],
    501
)

e1, e2 = np.meshgrid(
    x1_axis,
    x2_axis
)

exp_candidates = np.column_stack([
    e1.ravel(),
    e2.ravel()
])

distance, _ = tree.query(
    exp_candidates,
    k=1
)

exp_candidates = exp_candidates[
    distance > 0.01
]

exp_mu, exp_sigma = gp.predict(
    exp_candidates,
    return_std=True
)

# ------------------------------------------------------------
# EI
# ------------------------------------------------------------

exp_EI = expected_improvement(
    exp_mu,
    exp_sigma,
    best_y_scaled,
    xi=0.0
)

ei_idx = np.argmax(exp_EI)

print("\nEXPANDED EI:")
print("candidate =", exp_candidates[ei_idx])
print("scaled mean =", exp_mu[ei_idx])
print("scaled std =", exp_sigma[ei_idx])
print("EI =", exp_EI[ei_idx])


# ------------------------------------------------------------
# Highest mean
# ------------------------------------------------------------

mean_idx = np.argmax(exp_mu)

print("\nEXPANDED highest mean:")
print("candidate =", exp_candidates[mean_idx])
print("scaled mean =", exp_mu[mean_idx])
print("scaled std =", exp_sigma[mean_idx])


# ------------------------------------------------------------
# UCB
# ------------------------------------------------------------

print("\nEXPANDED UCB:\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = exp_mu + beta * exp_sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", exp_candidates[idx],
        "\n scaled mean =", exp_mu[idx],
        "\n scaled std =", exp_sigma[idx],
        "\n UCB =", UCB[idx],
        "\n"
    )

Original half-widths: [0.05255692 0.01729457]
Expanded half-widths: [0.10511383 0.01729457]

Expanded lower bounds: [0.6259098 0.7157053]
Expanded upper bounds: [0.83613747 0.75029445]

EXPANDED EI:
candidate = [0.6259098  0.73721975]
scaled mean = 0.006782864633798458
scaled std = 0.011055810472190225
EI = 0.008607046275684534

EXPANDED highest mean:
candidate = [0.6259098 0.7370814]
scaled mean = 0.006789742465441118
scaled std = 0.011023412648128788

EXPANDED UCB:

beta=0.1 
 candidate = [0.6259098 0.7370814] 
 scaled mean = 0.006789742465441118 
 scaled std = 0.011023412648128788 
 UCB = 0.007892083730253997 

beta=0.25 
 candidate = [0.6259098  0.73715057] 
 scaled mean = 0.006787692660588873 
 scaled std = 0.011040652851338725 
 UCB = 0.009547855873423555 

beta=0.5 
 candidate = [0.6259098  0.73721975] 
 scaled mean = 0.006782864633798458 
 scaled std = 0.011055810472190225 
 UCB = 0.01231076986989357 

beta=1.0 
 candidate = [0.6259098  0.73735811] 
 scaled mean = 0.00676504631

In [4]:
# --------------------------------------------------
# Final Function 1 Week 9 selection
# --------------------------------------------------
#
# After one formal trust-region expansion, EI,
# highest mean and all UCB settings still hit the
# lower x1 boundary.
#
# We stop expanding to avoid chasing unreliable
# global GP extrapolation.
#
# beta = 0.5 is used as a moderate
# exploration-exploitation compromise.

beta = 0.5

UCB = exp_mu + beta * exp_sigma
final_idx = np.argmax(UCB)

week9_candidate = exp_candidates[final_idx]

print("Week 9 Function 1 candidate:")
print(week9_candidate)

print("\nScaled predicted mean:")
print(exp_mu[final_idx])

print("\nScaled predicted std:")
print(exp_sigma[final_idx])

print("\nUCB:")
print(UCB[final_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 1 candidate:
[0.6259098  0.73721975]

Scaled predicted mean:
0.006782864633798458

Scaled predicted std:
0.011055810472190225

UCB:
0.01231076986989357

Portal format:
0.625910-0.737220
